# 04 - Experiments And Improvements

This notebook defines the experiment plan. Training is disabled by default. It uses normal imports to inspect project code, and uses `subprocess` only for commands that would launch full scripts.

## Step 1 - Setup

In [ ]:
from pathlib import Path
import inspect
import json
import sys

def find_notebooks_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd / "salient-object-detection-cnn" / "notebooks"]
    for parent in cwd.parents:
        candidates.extend([parent / "notebooks", parent / "salient-object-detection-cnn" / "notebooks"])
    for candidate in candidates:
        if (candidate / "notebook_utils.py").exists():
            return candidate
    raise FileNotFoundError("Could not find notebooks/notebook_utils.py")

NOTEBOOKS_DIR = find_notebooks_dir()
sys.path.insert(0, str(NOTEBOOKS_DIR))

from notebook_utils import PROJECT_DIR, PYTHON, run_script

sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

## Step 2 - Safety Switch

`RUN_TRAINING` is intentionally `False`. With this setting, training/evaluation/visualization cells only print the subprocess command to run later.

In [ ]:
RUN_TRAINING = False

def show_or_run(script_name, *args):
    command = [PYTHON, script_name, *map(str, args)]
    if RUN_TRAINING:
        return run_script(script_name, *args)
    print("DRY RUN - training is disabled")
    print("Command to run later:")
    print(" ".join(command))
    return None

## Step 3 - Experiment Plan

| Run | Purpose | Current support in code | Notes |
|---|---|---|---|
| Baseline | Original encoder-decoder model | Supported | Uses `--model_type baseline`. |
| Experiment 1 | Add Dropout | Partial | `SmallUNet` has dropout, but the dropout value is fixed unless the code is extended. |
| Experiment 2 | Add BatchNorm | Partial | BatchNorm already exists in both current models, so a clean ablation needs a no-BatchNorm model. |
| Experiment 3 | Stronger augmentation | Partial | Augmentation exists in `data_loader.py`, but strength is not configurable from CLI yet. |
| Experiment 4 | Different loss weighting | Partial | Loss is fixed as `BCE + 0.5 * IoU`; proper testing needs configurable weights. |

## Step 4 - Shared Experiment Settings

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-3
NUM_WORKERS = 2
NUM_VISUALIZATION_SAMPLES = 10

COMMON_TRAIN_ARGS = [
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--epochs", EPOCHS,
    "--lr", LR,
    "--num_workers", NUM_WORKERS,
]

COMMON_EVAL_ARGS = [
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--num_workers", NUM_WORKERS,
]

print("Training will run:", RUN_TRAINING)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LR)

## Step 5 - Preflight: Inspect Current Implementation

In [ ]:
import torch.nn as nn

from sod_model import BaselineSODCNN, SmallUNet
from losses import combined_loss

for name, model in [("baseline", BaselineSODCNN()), ("unet_small", SmallUNet())]:
    batchnorm_count = sum(1 for module in model.modules() if isinstance(module, nn.BatchNorm2d))
    dropout_count = sum(1 for module in model.modules() if isinstance(module, nn.Dropout2d))
    print(f"{name}: BatchNorm2d layers={batchnorm_count}, Dropout2d layers={dropout_count}")

print("SmallUNet signature:", inspect.signature(SmallUNet))
print("combined_loss signature:", inspect.signature(combined_loss))

## Baseline - Original Model

In [ ]:
show_or_run("train.py", *COMMON_TRAIN_ARGS, "--model_type", "baseline")

### Baseline Evaluation And Visualization Commands

In [ ]:
show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "baseline",
    "--checkpoint", "checkpoints/best_model_baseline.pth",
)

show_or_run(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", "baseline",
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--num_samples", NUM_VISUALIZATION_SAMPLES,
)

## Experiment 1 - Add Dropout

Purpose: test whether dropout improves generalization by reducing overfitting.

Current status: `SmallUNet` includes `Dropout2d`, so this uses the existing dropout-capable architecture. For a stricter `dropout_0.3` experiment, add a `--dropout` CLI argument later.

In [ ]:
show_or_run("train.py", *COMMON_TRAIN_ARGS, "--model_type", "unet_small")

### Experiment 1 Evaluation And Visualization Commands

In [ ]:
show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "unet_small",
    "--checkpoint", "checkpoints/best_model_unet_small.pth",
)

show_or_run(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", "unet_small",
    "--checkpoint", "checkpoints/best_model_unet_small.pth",
    "--num_samples", NUM_VISUALIZATION_SAMPLES,
)

## Experiment 2 - Add BatchNorm

Purpose: test whether BatchNorm stabilizes training and improves segmentation quality.

Current status: this cannot be measured as a clean new experiment yet because the current baseline already uses `BatchNorm2d`. A valid BatchNorm experiment should compare a no-BatchNorm model against the same model with BatchNorm.

In [ ]:
print("No training command is run here because BatchNorm is already present in the current baseline.")
print("Recommended future commands after adding model_type='baseline_no_bn':")
print(f"{PYTHON} train.py --data_dir pre-processed --image_size {IMAGE_SIZE} --batch_size {BATCH_SIZE} --epochs {EPOCHS} --lr {LR} --model_type baseline_no_bn")
print(f"{PYTHON} train.py --data_dir pre-processed --image_size {IMAGE_SIZE} --batch_size {BATCH_SIZE} --epochs {EPOCHS} --lr {LR} --model_type baseline")

## Experiment 3 - Stronger Augmentation

Purpose: test whether stronger augmentation improves robustness on unseen DUTS-TE images.

Current status: augmentation exists in `data_loader.py`, but its strength is not configurable from `train.py` yet.

In [ ]:
from data_loader import PreprocessedDUTSDataset

print(inspect.getsource(PreprocessedDUTSDataset._apply_train_augmentations))
print("Recommended future setup:")
print("- Add an augmentation_strength option to data_loader.py/train.py.")
print("- Compare default augmentation vs stronger crop/rotation/brightness settings.")
print("- Keep epochs, batch size, learning rate, seed, and model architecture identical.")

## Experiment 4 - Different Loss Weighting

Purpose: test whether emphasizing IoU improves mask overlap compared with the default loss.

Current status: the loss is fixed as `BCE + 0.5 * IoU loss`. A true loss-weight experiment needs configurable loss weights in `losses.py` and `train.py`.

In [ ]:
print(inspect.getsource(combined_loss))

loss_weight_plan = [
    {"name": "default_loss", "bce_weight": 1.0, "iou_weight": 0.5},
    {"name": "iou_heavy", "bce_weight": 1.0, "iou_weight": 1.0},
    {"name": "bce_heavy", "bce_weight": 1.5, "iou_weight": 0.5},
]

for plan in loss_weight_plan:
    print(plan)

## Step 6 - Results Table Template

In [ ]:
metric_files = {
    "baseline": PROJECT_DIR / "outputs/metrics/test_metrics_baseline.json",
    "dropout": PROJECT_DIR / "outputs/metrics/test_metrics_dropout.json",
    "batchnorm": PROJECT_DIR / "outputs/metrics/test_metrics_batchnorm.json",
    "strong_aug": PROJECT_DIR / "outputs/metrics/test_metrics_strong_aug.json",
    "iou_heavy": PROJECT_DIR / "outputs/metrics/test_metrics_iou_heavy.json",
}

header = ["experiment", "iou", "precision", "recall", "f1_score", "mae", "status"]
print("\t".join(header))
for name, path in metric_files.items():
    if path.exists():
        metrics = json.loads(path.read_text())
        values = [name] + [f"{metrics[key]:.4f}" for key in header[1:-1]] + ["completed"]
    else:
        values = [name, "-", "-", "-", "-", "-", f"missing: {path.name}"]
    print("\t".join(values))

## Step 7 - Fair Comparison Checklist

- Use the same DUTS train/validation/test split for every run.
- Use the same image size, batch size, epoch count, learning rate, optimizer, and random seed unless that is the variable being tested.
- Save a separate checkpoint and metric JSON for each experiment.
- Include at least 3 to 5 visual examples per experiment.
- Report both quantitative results and qualitative failure cases.